In [ ]:
# Core
import os
import pandas as pd
import copy
import numpy as np
import pybedtools

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns
# Statistics
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import binomtest
from adjustText import adjust_text
from sklearn.metrics import precision_recall_curve
from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import ks_2samp


from scipy.stats import ks_2samp # Kolmogorov–Smirnov (KS)
from scipy.stats import linregress
from scipy.stats import gaussian_kde


In [ ]:
import os, sys
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
import const
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
from const import save_fig
const.set_plot_style()

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

# Load archaic data

In [ ]:
# Load archaic MPRA results
archaic_MPRA_output= pd.read_csv(f'ArchaicDerivedMPRA/all_AH_mpra_v3.tsv', 
                     sep='\t',
                     header=0)
archaic_MPRA_output = archaic_MPRA_output[['seq_id','logFC_osteo','logFC_npc',
                           'fdr.mad_adjusted_archaic_osteo','fdr.mad_adjusted_modern_osteo','fdr.comparative_osteo',
                           'fdr.mad_adjusted_archaic_npc','fdr.mad_adjusted_modern_npc','fdr.comparative_npc' ]]

# Add fields
archaic_MPRA_output['active_osteoblasts'] = (archaic_MPRA_output['fdr.mad_adjusted_archaic_osteo'] < 0.05)|(archaic_MPRA_output['fdr.mad_adjusted_modern_osteo'] < 0.05)
archaic_MPRA_output['diff_active_osteoblasts'] = archaic_MPRA_output['fdr.comparative_osteo'] < 0.05

archaic_MPRA_output['active_npc'] = (archaic_MPRA_output['fdr.mad_adjusted_modern_npc'] < 0.05)|(archaic_MPRA_output['fdr.mad_adjusted_archaic_npc'] < 0.05)
archaic_MPRA_output['diff_active_npc'] = archaic_MPRA_output['fdr.comparative_npc'] < 0.05

In [ ]:
# Load FIMO and PBM results
archaic_osteo_PBM = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/arhcaic_osteo_PBM.tsv', sep='\t')
archaic_osteo_PBM['motif_id_clean'] = archaic_osteo_PBM['motif_id_clean'].str.upper()

archaic_osteo_FIMO = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/arhcaic_osteo_FIMO.tsv', sep='\t')
archaic_osteo_FIMO['motif_id_clean'] = archaic_osteo_FIMO['motif_id_clean'].str.upper()

archaic_npc_PBM = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/arhcaic_npc_PBM.tsv', sep='\t')
archaic_npc_PBM['motif_id_clean'] = archaic_npc_PBM['motif_id_clean'].str.upper()

archaic_npc_FIMO = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/arhcaic_npc_FIMO.tsv', sep='\t')
archaic_npc_FIMO['motif_id_clean'] = archaic_npc_FIMO['motif_id_clean'].str.upper()

archaic_osteo = pd.concat([archaic_osteo_PBM, archaic_osteo_FIMO], axis = 0)
archaic_npc = pd.concat([archaic_npc_PBM, archaic_npc_FIMO], axis = 0)

In [ ]:
merged_archaic_df = pd.merge(archaic_osteo, archaic_MPRA_output, on='seq_id', how='outer')

PBM_arhciac_osteo_merged_df = pd.merge(archaic_osteo_PBM, archaic_MPRA_output, on='seq_id', how='outer')
FIMO_arhciac_osteo_merged_df = pd.merge(archaic_osteo_FIMO, archaic_MPRA_output, on='seq_id', how='outer')
PBM_arhciac_npc_merged_df = pd.merge(archaic_npc_PBM, archaic_MPRA_output, on='seq_id', how='outer')
FIMO_arhciac_npc_merged_df = pd.merge(archaic_npc_FIMO, archaic_MPRA_output, on='seq_id', how='outer')

# Load modern data

In [ ]:
# Load modern results
modern_MPRA_output= pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/intermediate_files/modern_MPRA_processed.tsv', 
                     sep='\t',
                     header=0)


In [ ]:
# Load FIMO and PBM results
modern_osteo_PBM = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/modern_osteo_PBM.tsv', sep='\t')
modern_osteo_PBM['motif_id_clean'] = modern_osteo_PBM['motif_id_clean'].str.upper()

modern_osteo_FIMO = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/modern_osteo_FIMO.tsv', sep='\t')
modern_osteo_FIMO['motif_id_clean'] = modern_osteo_FIMO['motif_id_clean'].str.upper()

modern_npc_PBM = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/modern_npc_PBM.tsv', sep='\t')
modern_npc_PBM['motif_id_clean'] = modern_npc_PBM['motif_id_clean'].str.upper()

modern_npc_FIMO = pd.read_csv(f'ArchaicDerivedMPRA/TF_analysis/results/modern_npc_FIMO.tsv', sep='\t')
modern_npc_FIMO['motif_id_clean'] = modern_npc_FIMO['motif_id_clean'].str.upper()

modern_osteo = pd.concat([modern_osteo_PBM, modern_osteo_FIMO], axis = 0)
modern_npc = pd.concat([modern_npc_PBM, modern_npc_FIMO], axis = 0)

In [ ]:
merged_modern_df = pd.merge(modern_osteo, modern_MPRA_output, on='seq_id', how='outer')

PBM_modern_osteo_merged_df = pd.merge(modern_osteo_PBM, modern_MPRA_output, on='seq_id', how='outer')
FIMO_modern_osteo_merged_df = pd.merge(modern_osteo_FIMO, modern_MPRA_output, on='seq_id', how='outer')
PBM_modern_npc_merged_df = pd.merge(modern_npc_PBM, modern_MPRA_output, on='seq_id', how='outer')
FIMO_modern_npc_merged_df = pd.merge(modern_npc_FIMO, modern_MPRA_output, on='seq_id', how='outer')

In [ ]:
PBM_all_osteo_merged_df = pd.concat(
    [PBM_modern_osteo_merged_df.assign(population="modern"), 
     PBM_arhciac_osteo_merged_df.assign(population="archaic")],
    axis=0,
    ignore_index=True
)

FIMO_all_osteo_merged_df = pd.concat(
    [FIMO_modern_osteo_merged_df.assign(population="modern"), 
     FIMO_arhciac_osteo_merged_df.assign(population="archaic")],
    axis=0,
    ignore_index=True
)

PBM_all_npc_merged_df = pd.concat(
    [PBM_modern_npc_merged_df.assign(population="modern"), 
     PBM_arhciac_npc_merged_df.assign(population="archaic")],
    axis=0,
    ignore_index=True
)

FIMO_all_npc_merged_df = pd.concat(
    [FIMO_modern_npc_merged_df.assign(population="modern"), 
     FIMO_arhciac_npc_merged_df.assign(population="archaic")],
    axis=0,
    ignore_index=True
)

In [ ]:
# get motif lists
PBM_all_osteo_motifs = PBM_all_osteo_merged_df['motif_id_clean'].dropna().unique().tolist()
FIMO_all_osteo_motifs = FIMO_all_osteo_merged_df['motif_id_clean'].dropna().unique().tolist()
PBM_all_npc_motifs = PBM_all_npc_merged_df['motif_id_clean'].dropna().unique().tolist()
FIMO_all_npc_motifs = FIMO_all_npc_merged_df['motif_id_clean'].dropna().unique().tolist()

In [ ]:
def keep_most_extreme_diff_binding(df):
    temp = df.copy()
    temp = temp.dropna(subset=["diff_binding_zscore"])
    temp["_abs_z"] = temp["diff_binding_zscore"].abs()
    temp = temp.sort_values(["seq_id", "_abs_z"], ascending=[True, False])
    temp = temp.drop_duplicates(subset="seq_id", keep="first")
    return temp.drop(columns="_abs_z").reset_index(drop=True)

def check_most_extreme_diff_binding(original_df, filtered_df):
    # 1) Check uniqueness of seq_id
    unique_ok = not filtered_df["seq_id"].duplicated().any()

    # 2) Expected absolute max per seq_id from original df
    expected = (
        original_df.assign(abs_z=original_df["diff_binding_zscore"].abs())
        .groupby("seq_id", as_index=False)["abs_z"]
        .max()
        .rename(columns={"abs_z": "expected_abs_z"})
    )

    # 3) Observed absolute z-score in filtered df
    observed = (
        filtered_df.assign(observed_abs_z=filtered_df["diff_binding_zscore"].abs())
        [["seq_id", "diff_binding_zscore", "observed_abs_z"]]
    )

    # 4) Compare
    check_df = expected.merge(observed, on="seq_id", how="outer")
    check_df["matches"] = check_df["expected_abs_z"] == check_df["observed_abs_z"]

    all_ok = unique_ok and check_df["matches"].all() and (len(filtered_df) == original_df["seq_id"].nunique())

    print(f"Unique seq_id check: {unique_ok}")
    print(f"Row count check: {len(filtered_df) == original_df['seq_id'].nunique()}")
    print(f"Extreme-value check: {check_df['matches'].all()}")
    print(f"Overall: {all_ok}")

    return check_df

## Fisher's exact - Bind/ Not Bind // Diff active oligos archaic / Diff active oligos modern

In [ ]:
def run_tf_fisher_test(relevant_df, relevant_tfs, diff_active_col,min_binding_threshold):
    results = []

    for tf in relevant_tfs:
        bind_df = relevant_df[relevant_df["motif_id_clean"] == tf]
        not_bind_df = relevant_df[relevant_df["motif_id_clean"] != tf]

        a = bind_df.loc[
            (bind_df[diff_active_col] == 1) & (bind_df["population"] == "archaic"),
            "seq_id"
        ].nunique()

        b = bind_df.loc[
            (bind_df[diff_active_col] == 1) & (bind_df["population"] == "modern"),
            "seq_id"
        ].nunique()

        c = not_bind_df.loc[
            (not_bind_df[diff_active_col] == 1) & (not_bind_df["population"] == "archaic"),
            "seq_id"
        ].nunique()

        d = not_bind_df.loc[
            (not_bind_df[diff_active_col] == 1) & (not_bind_df["population"] == "modern"),
            "seq_id"
        ].nunique()

        table = [[a, b], [c, d]]
        oddsratio, pvalue = fisher_exact(table)

        results.append({
            "TF": tf,
            "a_bind_tf_and_diff_active_archaic": a,
            "b_bind_tf_and_diff_active_modern": b,
            "c_not_bind_tf_and_diff_active_archaic": c,
            "d_not_bind_tf_and_diff_active_modern": d,
            "odds_ratio": oddsratio,
            "pvalue": pvalue
        })

    fisher_df = pd.DataFrame(results)

    sufficient_binding = (
        fisher_df["a_bind_tf_and_diff_active_archaic"] +
        fisher_df["b_bind_tf_and_diff_active_modern"]
    ) >= min_binding_threshold

    fdr_values = np.full(len(fisher_df), np.nan)
    if sufficient_binding.any():
        fdr_values[sufficient_binding] = multipletests(
            fisher_df.loc[sufficient_binding, "pvalue"].values,
            method="fdr_bh"
        )[1]

    fisher_df["pval_fdr"] = fdr_values
    fisher_df = fisher_df.sort_values("pval_fdr").reset_index(drop=True)

    return fisher_df

### FIMO NPCs

In [ ]:
fisher_params = {
    "relevant_df": FIMO_all_npc_merged_df.copy(),
    "relevant_tfs": FIMO_all_npc_motifs,
    "diff_active_col": "diff_active_npc",
    "min_binding_threshold": 10
}

fisher_npcs_FIMO_df = run_tf_fisher_test(**fisher_params)
fisher_npcs_FIMO_df

### PBM NPCs

In [ ]:
fisher_params = {
    "relevant_df": PBM_all_npc_merged_df.copy(),
    "relevant_tfs": PBM_all_npc_motifs,
    "diff_active_col": "diff_active_npc",
    "min_binding_threshold": 10
}

fisher_npcs_PBM_df = run_tf_fisher_test(**fisher_params)
fisher_npcs_PBM_df

### Osteo FIMO

In [ ]:
fisher_params = {
    "relevant_df":FIMO_all_osteo_merged_df.copy(),
    "relevant_tfs": FIMO_all_osteo_motifs,
    "diff_active_col": "diff_active_osteoblasts",
    "min_binding_threshold": 10
}

fisher_osteoblasts_FIMO_df = run_tf_fisher_test(**fisher_params)
fisher_osteoblasts_FIMO_df

### Osteo PBM

In [ ]:
fisher_params = {
    "relevant_df":PBM_all_osteo_merged_df.copy(),
    "relevant_tfs": PBM_all_osteo_motifs,
    "diff_active_col": "diff_active_osteoblasts",
    "min_binding_threshold": 10
}

fisher_osteoblasts_PBM_df = run_tf_fisher_test(**fisher_params)
fisher_osteoblasts_PBM_df

# diff active up vs down

In [ ]:
def run_tf_fisher_up_down_by_population(relevant_df, relevant_tfs, diff_active_col, logFC_col,min_binding_threshold):
    results = []

    for tf in relevant_tfs:
        bind_df = relevant_df[
            (relevant_df["motif_id_clean"] == tf) &
            (relevant_df[diff_active_col] == 1)
        ].copy()

        a = bind_df.loc[
            (bind_df["population"] == "archaic") & (bind_df[logFC_col] > 0),
            "seq_id"
        ].nunique()

        b = bind_df.loc[
            (bind_df["population"] == "archaic") & (bind_df[logFC_col] < 0),
            "seq_id"
        ].nunique()

        c = bind_df.loc[
            (bind_df["population"] == "modern") & (bind_df[logFC_col] > 0),
            "seq_id"
        ].nunique()

        d = bind_df.loc[
            (bind_df["population"] == "modern") & (bind_df[logFC_col] < 0),
            "seq_id"
        ].nunique()

        table = [[a, b], [c, d]]
        oddsratio, pvalue = fisher_exact(table)

        results.append({
            "TF": tf,
            "a_archaic_up": a,
            "b_archaic_down": b,
            "c_modern_up": c,
            "d_modern_down": d,
            "odds_ratio": oddsratio,
            "pvalue": pvalue
        })

    fisher_df = pd.DataFrame(results)

    sufficient_binding = (
        fisher_df["a_archaic_up"] +
        fisher_df["b_archaic_down"] +
        fisher_df["c_modern_up"] +
        fisher_df["d_modern_down"]
    ) >= min_binding_threshold

    fdr_values = np.full(len(fisher_df), np.nan)
    if sufficient_binding.any():
        fdr_values[sufficient_binding] = multipletests(
            fisher_df.loc[sufficient_binding, "pvalue"].values,
            method="fdr_bh"
        )[1]

    fisher_df["pval_fdr"] = fdr_values
    fisher_df = fisher_df.sort_values("pval_fdr").reset_index(drop=True)

    return fisher_df

## NPCs FIMO

In [ ]:
fisher_params = {
    "relevant_df": FIMO_all_npc_merged_df.copy(),
    "relevant_tfs": FIMO_all_npc_motifs,
    "diff_active_col": "diff_active_npc",
    "logFC_col": "logFC_npc",
    "min_binding_threshold": 20
}
fisher_up_down_df_FIMO_NPCs = run_tf_fisher_up_down_by_population(**fisher_params)
fisher_up_down_df_FIMO_NPCs

## NPCs PBM

In [ ]:
fisher_params = {
    "relevant_df": PBM_all_npc_merged_df.copy(),
    "relevant_tfs": PBM_all_npc_motifs,
    "diff_active_col": "diff_active_npc",
    "logFC_col": "logFC_npc",
    "min_binding_threshold": 20
}
fisher_up_down_df_PBM_NPCs = run_tf_fisher_up_down_by_population(**fisher_params)
fisher_up_down_df_PBM_NPCs

## Osteo FIMO

In [ ]:
fisher_params = {
    "relevant_df": FIMO_all_osteo_merged_df.copy(),
    "relevant_tfs": FIMO_all_osteo_motifs,
    "diff_active_col": "diff_active_osteoblasts",
    "logFC_col": "logFC_osteo",
    "min_binding_threshold": 20
}
fisher_up_down_df_FIMO_Osteo = run_tf_fisher_up_down_by_population(**fisher_params)
fisher_up_down_df_FIMO_Osteo

In [ ]:
fisher_params = {
    "relevant_df": PBM_all_osteo_merged_df.copy(),
    "relevant_tfs": PBM_all_osteo_motifs,
    "diff_active_col": "diff_active_osteoblasts",
    "logFC_col": "logFC_osteo",
    "min_binding_threshold": 20
}
fisher_up_down_df_PBM_Osteo = run_tf_fisher_up_down_by_population(**fisher_params)
fisher_up_down_df_PBM_Osteo


## Osteoblasts - Distribution of logFC in archaic vs modern

In [ ]:
relevant_df = FIMO_all_npc_merged_df.copy()
relevant_tfs = FIMO_all_npc_motifs
logFC_col = 'logFC_npc'

results = []

for tf in relevant_tfs:
    bind_mask = (relevant_df["motif_id_clean"] == tf)
    bind_df = relevant_df.loc[bind_mask]
    
    # Separate by population
    bind_archaic = bind_df[bind_df['population'] == 'archaic']
    bind_modern = bind_df[bind_df['population'] == 'modern']

    # helper: run U-test on a column, return stats (or NaNs if insufficient)
    def utest(col, x_data, y_data):
        x = x_data[col].dropna().astype(float).values
        y = y_data[col].dropna().astype(float).values

        # Need at least 2 values per group to be meaningful
        if (len(x) < 2) or (len(y) < 2):
            return np.nan, np.nan, len(x), len(y), np.nan, np.nan

        # two-sided by default; method='auto' picks exact/asymptotic as appropriate
        U, p = mannwhitneyu(x, y, alternative="two-sided", method="auto")

        # nice to also report medians (effect direction)
        return U, p, len(x), len(y), np.median(x), np.median(y)

    Uo, po, nbo, nmo, med_bo, med_mo = utest(logFC_col, bind_archaic, bind_modern)

    results.append({
        "TF": tf,

        # osteo
        "n_bind_archaic": nbo,
        "n_bind_modern": nmo,
        "median_bind_archaic_logFC": med_bo,
        "median_bind_modern_logFC": med_mo,
        "U": Uo,
        "pvalue": po,
    })

utest_df = pd.DataFrame(results)

# Only calculate FDR on TFs with sufficient binding events
min_binding_threshold = 10
sufficient_binding = (
    utest_df['n_bind_archaic'] + utest_df['n_bind_modern'] >= min_binding_threshold
)

# initialize column
utest_df['pval_fdr'] = np.nan

# run BH only on valid pvalues among sufficient rows
mask = sufficient_binding & utest_df['pvalue'].notna()
if mask.any():
    pvals = utest_df.loc[mask, 'pvalue'].to_numpy()
    fdrs = multipletests(pvals, method='fdr_bh')[1]
    utest_df.loc[mask, 'pval_fdr'] = fdrs



# direction / effect summary (difference in medians: archaic - modern)
utest_df["delta_median"] = utest_df["median_bind_archaic_logFC"] - utest_df["median_bind_modern_logFC"]

# sort placing smallest osteo FDR first (NaNs last)
#utest_df = utest_df.sort_values("pval_fdr", na_position="last").reset_index(drop=True)

utest_df

In [ ]:
PBM_all_osteo_merged_df

In [ ]:
TF = 'IRF2'
logFC_col = 'logFC_osteo'

relevant_df = PBM_all_osteo_merged_df[
    PBM_all_osteo_merged_df['diff_active_osteoblasts'] == 1
]

tf_df = relevant_df[relevant_df['motif_id_clean'] == TF]

arch = tf_df.loc[tf_df['population'] == 'archaic', logFC_col].dropna()
mod  = tf_df.loc[tf_df['population'] == 'modern',  logFC_col].dropna()

# shared bins from -2 to 2
bins = np.linspace(-2, 2, 21)   # 20 equal-width bins

plt.figure(figsize=(8, 6))
sns.histplot(
    arch,
    bins=bins,
    color='tab:blue',
    label='archaic',
    stat='density',
    alpha=0.5,
    kde=False
)
sns.histplot(
    mod,
    bins=bins,
    color='tab:orange',
    label='modern',
    stat='density',
    alpha=0.5,
    kde=False
)

#plt.xlim(-3, 3)
plt.xlabel(logFC_col)
plt.ylabel('Density')
plt.title(f'{TF} {logFC_col} distribution: archaic vs modern')
plt.legend()
plt.tight_layout()

save_fig(plt, 'IRF2_logFC_osteo_distrbution_archaic_vs_modern', 'ArchaicDerivedMPRA/TF_analysis/figures/archaic_vs_modern/')
plt.show()

In [ ]:
TF = 'HES1'
logFC_col = 'logFC_osteo'

relevant_df = PBM_all_osteo_merged_df[
    PBM_all_osteo_merged_df['diff_active_osteoblasts'] == 1
]

tf_df = relevant_df[relevant_df['motif_id_clean'] == TF]

arch = tf_df.loc[tf_df['population'] == 'archaic', logFC_col].dropna()
mod  = tf_df.loc[tf_df['population'] == 'modern',  logFC_col].dropna()

# shared bins from -2 to 2
bins = np.linspace(-2, 2, 21)   # 20 equal-width bins

plt.figure(figsize=(8, 6))
sns.histplot(
    arch,
    bins=bins,
    color='tab:blue',
    label='archaic',
    stat='density',
    alpha=0.5,
    kde=False
)
sns.histplot(
    mod,
    bins=bins,
    color='tab:orange',
    label='modern',
    stat='density',
    alpha=0.5,
    kde=False
)

#plt.xlim(-3, 3)
plt.xlabel(logFC_col)
plt.ylabel('Density')
plt.title(f'{TF} {logFC_col} distribution: archaic vs modern')
plt.legend()
plt.tight_layout()

save_fig(plt, 'HES1_logFC_osteo_distrbution_archaic_vs_modern', 'ArchaicDerivedMPRA/TF_analysis/figures/archaic_vs_modern/')
plt.show()

# Comparison diff binding TFs in diff active oligos vs active oligos

In [ ]:
print(archaic_MPRA_output['diff_active_npc'].sum()+modern_MPRA_output['diff_active_npc'].sum())
print(archaic_MPRA_output['diff_active_osteoblasts'].sum()+modern_MPRA_output['diff_active_osteoblasts'].sum())

In [ ]:
def keep_most_extreme_diff_binding(df):
    temp = df.copy()
    temp = temp.dropna(subset=["diff_binding_zscore"])
    temp["_abs_z"] = temp["diff_binding_zscore"].abs()
    temp = temp.sort_values(["seq_id", "_abs_z"], ascending=[True, False])
    temp = temp.drop_duplicates(subset="seq_id", keep="first")
    return temp.drop(columns="_abs_z").reset_index(drop=True)

def check_most_extreme_diff_binding(original_df, filtered_df):
    # 1) Check uniqueness of seq_id
    unique_ok = not filtered_df["seq_id"].duplicated().any()

    # 2) Expected absolute max per seq_id from original df
    expected = (
        original_df.assign(abs_z=original_df["diff_binding_zscore"].abs())
        .groupby("seq_id", as_index=False)["abs_z"]
        .max()
        .rename(columns={"abs_z": "expected_abs_z"})
    )

    # 3) Observed absolute z-score in filtered df
    observed = (
        filtered_df.assign(observed_abs_z=filtered_df["diff_binding_zscore"].abs())
        [["seq_id", "diff_binding_zscore", "observed_abs_z"]]
    )

    # 4) Compare
    check_df = expected.merge(observed, on="seq_id", how="outer")
    check_df["matches"] = check_df["expected_abs_z"] == check_df["observed_abs_z"]

    all_ok = unique_ok and check_df["matches"].all() and (len(filtered_df) == original_df["seq_id"].nunique())

    print(f"Unique seq_id check: {unique_ok}")
    print(f"Row count check: {len(filtered_df) == original_df['seq_id'].nunique()}")
    print(f"Extreme-value check: {check_df['matches'].all()}")
    print(f"Overall: {all_ok}")

    return check_df

def plot_abs_diff_binding_ecdf_by_diff_activity(
    filtered_df,
    active_col="active_in_cell_type",
    diff_active_col="diff_active_in_cell_type",
    zscore_col="diff_binding_zscore",
    ax=None,
):
    """
    Filter to active enhancers/sites only, then plot two ECDFs of the absolute
    diff_binding_zscore:
      - diff active   (diff_active_col == True)
      - not diff active (diff_active_col == False)

    Parameters
    ----------
    filtered_df : pandas.DataFrame
    active_col : str
        Boolean column used to keep only True rows.
    diff_active_col : str
        Boolean column used to split rows into two groups.
    zscore_col : str
        Numeric column to plot in absolute value.
    ax : matplotlib.axes.Axes or None
        Existing axis to plot on. If None, creates a new figure.

    Returns
    -------
    fig, ax, plot_df
        fig : matplotlib Figure
        ax : matplotlib Axes
        plot_df : filtered dataframe used for plotting
    """
    plot_df = filtered_df.copy()

    # keep only active rows
    plot_df = plot_df[plot_df[active_col] == True].copy()

    # remove missing values just in case
    plot_df = plot_df.dropna(subset=[diff_active_col, zscore_col]).copy()

    # absolute z-score
    plot_df["abs_diff_binding_zscore"] = plot_df[zscore_col].abs()

    # split groups
    diff_active_vals = np.sort(
        plot_df.loc[plot_df[diff_active_col] == True, "abs_diff_binding_zscore"].to_numpy()
    )
    not_diff_active_vals = np.sort(
        plot_df.loc[plot_df[diff_active_col] == False, "abs_diff_binding_zscore"].to_numpy()
    )

    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    else:
        fig = ax.figure

    # ECDF helper
    def ecdf_y(n):
        return np.arange(1, n + 1) / n if n > 0 else np.array([])

    if len(diff_active_vals) > 0:
        ax.step(
            diff_active_vals,
            ecdf_y(len(diff_active_vals)),
            where="post",
            label=f"Diff active (n={len(diff_active_vals)})",
            linewidth=2,
        )

    if len(not_diff_active_vals) > 0:
        ax.step(
            not_diff_active_vals,
            ecdf_y(len(not_diff_active_vals)),
            where="post",
            label=f"Not diff active (n={len(not_diff_active_vals)})",
            linewidth=2,
        )

    ax.set_xlabel("|diff_binding_zscore|")
    ax.set_ylabel("ECDF")
    ax.set_title("ECDF of absolute diff_binding_zscore")
    ax.legend(frameon=False)
    plt.tight_layout()

    return fig, ax, plot_df

## NPCs FIMO

In [ ]:
filtered_FIMO_all_npc_merged_df = keep_most_extreme_diff_binding(FIMO_all_npc_merged_df)
check_df = check_most_extreme_diff_binding(FIMO_all_npc_merged_df.dropna(subset=["diff_binding_zscore"], inplace=False), filtered_FIMO_all_npc_merged_df)

filtered_FIMO_all_npc_merged_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/ecdfs/filtered_FIMO_all_npc_merged_df.tsv', sep='\t', index=False)

In [ ]:
fig, ax, plot_df, ks_result = plot_abs_diff_binding_ecdf_by_diff_activity(filtered_FIMO_all_npc_merged_df,
active_col = "active_npc", diff_active_col = "diff_active_npc",zscore_col = "diff_binding_zscore")
save_fig(plt, 'NPC_FIMO_zscore_correlation_osteobalsts', 'ArchaicDerivedMPRA/TF_analysis/figures/ecdfs')

plt.show()
print(ks_result)



## NPCs PBM

In [ ]:
filtered_PBM_all_npc_merged_df = keep_most_extreme_diff_binding(PBM_all_npc_merged_df)
check_df = check_most_extreme_diff_binding(PBM_all_npc_merged_df.dropna(subset=["diff_binding_zscore"], inplace=False), filtered_PBM_all_npc_merged_df)
filtered_PBM_all_npc_merged_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/ecdfs/filtered_PBM_all_npc_merged_df.tsv', sep='\t', index=False)


In [ ]:
fig, ax, plot_df,ks_result = plot_abs_diff_binding_ecdf_by_diff_activity(filtered_PBM_all_npc_merged_df,
active_col = "active_npc", diff_active_col = "diff_active_npc",zscore_col = "diff_binding_zscore")
save_fig(plt, 'NPC_PBM_zscore_correlation_osteobalsts', 'ArchaicDerivedMPRA/TF_analysis/figures/ecdfs')
plt.show()
print(ks_result)

## Osteo FIMO

In [ ]:
filtered_FIMO_all_osteo_merged_df = keep_most_extreme_diff_binding(FIMO_all_osteo_merged_df)
check_df = check_most_extreme_diff_binding(FIMO_all_osteo_merged_df.dropna(subset=["diff_binding_zscore"], inplace=False), filtered_FIMO_all_osteo_merged_df)
filtered_FIMO_all_osteo_merged_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/ecdfs/filtered_FIMO_all_osteo_merged_df.tsv', sep='\t', index=False)



In [ ]:
fig, ax, plot_df,ks_result = plot_abs_diff_binding_ecdf_by_diff_activity(filtered_FIMO_all_osteo_merged_df,
active_col = "active_osteoblasts", diff_active_col = "diff_active_osteoblasts",zscore_col = "diff_binding_zscore")
save_fig(plt, 'osteo_FIMO_zscore_correlation_osteobalsts', 'ArchaicDerivedMPRA/TF_analysis/figures/ecdfs')

plt.show()
print(ks_result)

## Osteo PBM

In [ ]:
filtered_PBM_all_osteo_merged_df = keep_most_extreme_diff_binding(PBM_all_osteo_merged_df)
check_df = check_most_extreme_diff_binding(PBM_all_osteo_merged_df.dropna(subset=["diff_binding_zscore"], inplace=False), filtered_PBM_all_osteo_merged_df)
filtered_PBM_all_osteo_merged_df.to_csv('ArchaicDerivedMPRA/TF_analysis/for_ryder/ecdfs/filtered_PBM_all_osteo_merged_df.tsv', sep='\t', index=False)



In [ ]:
fig, ax, plot_df,ks_result = plot_abs_diff_binding_ecdf_by_diff_activity(filtered_PBM_all_osteo_merged_df,
active_col = "active_osteoblasts", diff_active_col = "diff_active_osteoblasts",zscore_col = "diff_binding_zscore")
save_fig(plt, 'osteo_PBM_zscore_correlation_osteobalsts', 'ArchaicDerivedMPRA/TF_analysis/figures/ecdfs')

plt.show()
print(ks_result)